In [1]:
import numpy as np
import pandas as pd
import torch
import wandb
from dotenv import load_dotenv
from torch.utils.data import DataLoader

from soamp.data.factory import build_dataset
from soamp.data.torch_dataset import LABEL_TO_INT
from soamp.engine.class_balancing import resolve_pos_weight
from soamp.engine.metrics import compute_binary_metrics, logits_to_predictions
from soamp.engine.tracking import build_tracker, watch_model
from soamp.engine.trainer import Trainer
from soamp.model.factory import build_model

load_dotenv()

data_folder_path = "/Users/lukajin/PycharmProjects/soamp/data/"
thresholds_csv_path = "/Users/lukajin/PycharmProjects/soamp/config/thresholds/organism_thresholds.csv"

In [2]:
df = pd.read_csv(data_folder_path + "mic_classification_dataset.csv")
train_df = df[df["split"] != "test"]
train_df["organism"].value_counts()

organism
Escherichia coli          6714
Staphylococcus aureus     5981
Pseudomonas aeruginosa    4348
Name: count, dtype: int64

In [3]:
train_df.head()

,peptide_id,sequence,smiles,organism,ncbi_taxon_id_if_available,mic_value_uM,mic_type,has_noncanonical,label,split
0,10,LFIFFF,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,Staphylococcus aureus,1280,15.60,censored,False,active,train
1,11,RVKRVWPLVIRTVIAGYNLYRAIKKK,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Escherichia coli,562,2.24,averaged,False,active,train
2,11,RVKRVWPLVIRTVIAGYNLYRAIKKK,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Pseudomonas aeruginosa,287,5.06,averaged,False,active,train
3,11,RVKRVWPLVIRTVIAGYNLYRAIKKK,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Staphylococcus aureus,1280,1.11,averaged,False,active,train
8,15,DSHAKRHHGYKRKFHEKHHSHRGY,C[C@H](NC(=O)[C@H](Cc1cnc[nH]1)NC(=O)[C@H](CO)...,Staphylococcus aureus,1280,132.00,censored,False,inactive,train


In [4]:
train_folds = pd.read_csv(data_folder_path + "train_folds_leiden.csv")
train_folds.head()

,node_id,community,sequence,fold_id
0,0,26,LFIFFF,4
1,1,51,RVKRVWPLVIRTVIAGYNLYRAIKKK,0
2,2,415,DSHAKRHHGYKRKFHEKHHSHRGY,0
3,3,416,ENREVPPGFTALIKTLRKCKII,3
4,4,89,GMASKAGAIAGKIAKVALKAL,3


In [5]:
fold_columns = ["sequence", "fold_id"]
row_columns = ["sequence", "peptide_id", "smiles", "organism", "has_noncanonical", "label"]

In [6]:
train_df = pd.merge(
    train_folds[fold_columns],
    train_df[row_columns],
    on="sequence",
    how="inner",
)

In [7]:
train_df['fold_id'].value_counts()

fold_id
4    3627
3    3575
1    3343
2    3259
0    3239
Name: count, dtype: int64

In [8]:
train_df.head()

,sequence,fold_id,peptide_id,smiles,organism,has_noncanonical,label
0,LFIFFF,4,10,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,Staphylococcus aureus,False,active
1,RVKRVWPLVIRTVIAGYNLYRAIKKK,0,11,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Escherichia coli,False,active
2,RVKRVWPLVIRTVIAGYNLYRAIKKK,0,11,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Pseudomonas aeruginosa,False,active
3,RVKRVWPLVIRTVIAGYNLYRAIKKK,0,11,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Staphylococcus aureus,False,active
4,RVKRVWPLVIRTVIAGYNLYRAIKKK,0,9526,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)[C@H](CCCN...,Escherichia coli,False,active


In [9]:
train_df.shape

(17043, 7)

In [10]:
EPOCHS = 5
SEED = 42
BATCH_SIZE = 64
LEARNING_RATE = 1e-3

# Swapped in for this run: PeptideCLM (768-dim frozen transformer embedding)
# for peptide representation, genome k-mer composition (340-dim, LLAMP-style)
# for organism representation, and the attention-fusion architecture (which
# projects both representations to a shared PROJECTION_DIM before
# self-attending across them -- see src/soamp/model/attention_fusion.py --
# so their very different raw dimensionalities, 768 vs 340, don't bias the
# model toward one representation the way a naive concat would).
PEPTIDE_METHOD = "peptideclm_embedding"
ORGANISM_METHOD = "kmer_composition"
ARCHITECTURE = "attention_fusion_classifier"
PROJECTION_DIM = 128
NUM_ATTENTION_HEADS = 4
NUM_ATTENTION_LAYERS = 1
HIDDEN_DIMS = [64, 32]  # MLP head after the attended 2*PROJECTION_DIM concat

# Transcribed by hand from scripts/EDA/generate_leiden_train_folds.ipynb --
# that notebook builds train_folds_leiden.csv and isn't re-run here, so keep
# this in sync manually if its params ever change.
FOLD_GENERATION_METADATA = {
    "method": "sequence-identity graph (blosum45 alignment) + Leiden community "
              "detection; folds = greedy cumulative-count binning of communities "
              "into n_folds groups",
    "identity_threshold": 0.60,
    "substitution_matrix": "blosum45",
    "gap_open": 5,
    "gap_extension": 1,
    "leiden_n_iterations": -1,
    "leiden_seed": 42,
    "n_folds": 5,
    "source_notebook": "scripts/EDA/generate_leiden_train_folds.ipynb",
}

In [11]:
# build_dataset(row_groups=...) recomputes peptide featurization from
# scratch on every call (fresh per fold, since a fold's fit/val partition
# differs). That's cheap for rdkit_descriptors but not for PeptideCLM: ~79ms/
# peptide on this CPU-only Mac (no CUDA, MPS unused by the featurizer today)
# -- with ~8.6k unique peptides across train_df, one full pass is ~11 min,
# and the 5-fold loop below would otherwise redo it 5x (~57 min total).
#
# PeptideCLM embeddings are frozen/deterministic and don't depend on which
# rows are "fit" vs "val" for a given fold -- unlike the scaler/vocab, which
# genuinely must be fit per-fold to avoid leakage. So caching by peptide_id
# across folds is exact, not an approximation: every fold still gets the
# identical vector it would have gotten from a fresh compute. This patches
# PeptideCLMFeaturizer.transform in place (only in this notebook process,
# not src/) to memoize on peptide_id.
from soamp.features.peptide_featurizers import PeptideCLMFeaturizer

_peptide_clm_cache: dict[str, dict] = {}
_uncached_peptide_clm_transform = PeptideCLMFeaturizer.transform


def _cached_peptide_clm_transform(self, unique_peptides):
    to_compute = [p for p in unique_peptides if p["peptide_id"] not in _peptide_clm_cache]
    if to_compute:
        for row in _uncached_peptide_clm_transform(self, to_compute):
            _peptide_clm_cache[row["peptide_id"]] = row
    return [_peptide_clm_cache[p["peptide_id"]] for p in unique_peptides]


if PEPTIDE_METHOD == "peptideclm_embedding":
    PeptideCLMFeaturizer.transform = _cached_peptide_clm_transform

In [12]:
bundle = build_dataset(
    row_groups={"fit": train_df.to_dict("records")},
    peptide_method=PEPTIDE_METHOD,
    organism_method=ORGANISM_METHOD,
)

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

[transformers] RoFormerModel LOAD REPORT from: aaronfeller/PeptideCLM-23M-all
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
run = build_tracker(
    bundle,
    hyperparams={
        "architecture": ARCHITECTURE,
        "epochs": EPOCHS,
        "hidden_dims": HIDDEN_DIMS,
        "projection_dim": PROJECTION_DIM,
        "num_attention_heads": NUM_ATTENTION_HEADS,
        "num_attention_layers": NUM_ATTENTION_LAYERS,
        "seed": SEED,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "fold_generation": FOLD_GENERATION_METADATA,
    },
    project="soamp",
    job_type="kfold_cv",
    mode="offline"
)
dict(run.config)

wandb: Tracking run with wandb version 0.28.2


wandb: W&B syncing is set to `offline` in this directory. Run `wandb online` or set WANDB_MODE=online to enable cloud syncing.
wandb: Run data is saved locally in /Users/lukajin/PycharmProjects/soamp/scripts/EDA/wandb/offline-run-20260909_040853-hvii60i5
wandb: View this run in the terminal with `wandb leet`


{'architecture': 'attention_fusion_classifier',
 'epochs': 5,
 'hidden_dims': [64, 32],
 'projection_dim': 128,
 'num_attention_heads': 4,
 'num_attention_layers': 1,
 'seed': 42,
 'batch_size': 64,
 'learning_rate': 0.001,
 'fold_generation': {'gap_open': 5,
  'leiden_n_iterations': -1,
  'substitution_matrix': 'blosum45',
  'source_notebook': 'scripts/EDA/generate_leiden_train_folds.ipynb',
  'leiden_seed': 42,
  'identity_threshold': 0.6,
  'n_folds': 5,
  'gap_extension': 1,
  'method': 'sequence-identity graph (blosum45 alignment) + Leiden community detection; folds = greedy cumulative-count binning of communities into n_folds groups'},
 'peptide_method': 'peptideclm_embedding',
 'peptide_feature_dim': 768,
 'descriptor_names': ['dim_0',
  'dim_1',
  'dim_2',
  'dim_3',
  'dim_4',
  'dim_5',
  'dim_6',
  'dim_7',
  'dim_8',
  'dim_9',
  'dim_10',
  'dim_11',
  'dim_12',
  'dim_13',
  'dim_14',
  'dim_15',
  'dim_16',
  'dim_17',
  'dim_18',
  'dim_19',
  'dim_20',
  'dim_21',
  'd

In [14]:
thresholds_df = pd.read_csv(thresholds_csv_path)
organisms_used = sorted(train_df["organism"].unique())
organism_thresholds = thresholds_df[
    thresholds_df["match_key"].isin(organisms_used) & (thresholds_df["level"] == "species")
][["match_key", "active_threshold_uM", "inactive_threshold_uM", "source"]]

thresholds_artifact = wandb.Artifact(name="organism_activity_thresholds", type="dataset")
thresholds_artifact.add(wandb.Table(dataframe=organism_thresholds), "organism_thresholds")
run.log_artifact(thresholds_artifact)
organism_thresholds

,match_key,active_threshold_uM,inactive_threshold_uM,source
1,Escherichia coli,32.0,128.0,user-specified 2026-08-15
3,Staphylococcus aureus,32.0,128.0,user-specified 2026-08-15
5,Pseudomonas aeruginosa,32.0,128.0,user-specified 2026-08-15


In [15]:
def train_and_evaluate(row_groups, eval_groups=("fit", "val"), epochs=5, seed=42,
                        peptide_method="rdkit_descriptors", organism_method="vocab_embedding",
                        architecture="baseline_classifier", architecture_kwargs=None,
                        batch_size=64, learning_rate=1e-3, watch=False):
    torch.manual_seed(seed)
    bundle = build_dataset(row_groups=row_groups, peptide_method=peptide_method, organism_method=organism_method)
    model = build_model(bundle, architecture=architecture, **(architecture_kwargs or {}))
    if watch:
        # Watches the real model about to be trained -- gradient/parameter
        # histograms and the computation graph populate from its actual
        # forward/backward passes below, not a throwaway preview model.
        watch_model(model)

    fit_loader = DataLoader(bundle.datasets["fit"], batch_size=batch_size, shuffle=True)

    fit_labels = [LABEL_TO_INT[row["label"]] for row in row_groups["fit"]]
    pos_weight = resolve_pos_weight("auto", None, fit_labels)
    loss_fn = torch.nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor(pos_weight) if pos_weight is not None else None
    )
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    trainer = Trainer(model, optimizer, loss_fn)

    for _ in range(epochs):
        trainer.train_epoch(fit_loader)

    # Each eval_loader has shuffle=False, so its eval_out arrays line up 1:1
    # with row_groups[group] in order -- safe to reconstruct a per-row table.
    metrics_by_group = {}
    group_frames = []
    for group in eval_groups:
        eval_loader = DataLoader(bundle.datasets[group], batch_size=batch_size)
        eval_out = trainer.evaluate(eval_loader)
        metrics_by_group[group] = compute_binary_metrics(eval_out["logits"], eval_out["labels"])

        probs = 1.0 / (1.0 + np.exp(-eval_out["logits"]))
        group_df = pd.DataFrame(row_groups[group]).reset_index(drop=True)
        group_df["true_label_int"] = eval_out["labels"].astype(int)
        group_df["pred_label_int"] = logits_to_predictions(eval_out["logits"])
        group_df["logit"] = eval_out["logits"]
        group_df["prob"] = probs
        group_df["correct"] = group_df["true_label_int"] == group_df["pred_label_int"]
        group_df["eval_group"] = group
        group_frames.append(group_df)

    results_df = pd.concat(group_frames, ignore_index=True)
    return metrics_by_group, results_df

### K-fold CV over the Leiden clusters

In [16]:
fold_results = []
fold_metrics_rows = []
for fold_id in sorted(train_df["fold_id"].unique()):
    row_groups = {
        "fit": train_df[train_df["fold_id"] != fold_id].to_dict("records"),
        "val": train_df[train_df["fold_id"] == fold_id].to_dict("records"),
    }
    metrics_by_group, results_df = train_and_evaluate(
        row_groups, eval_groups=("fit", "val"),
        epochs=EPOCHS, seed=SEED,
        peptide_method=PEPTIDE_METHOD, organism_method=ORGANISM_METHOD,
        architecture=ARCHITECTURE,
        architecture_kwargs={
            "projection_dim": PROJECTION_DIM,
            "num_attention_heads": NUM_ATTENTION_HEADS,
            "num_attention_layers": NUM_ATTENTION_LAYERS,
            "hidden_dims": HIDDEN_DIMS,
        },
        batch_size=BATCH_SIZE, learning_rate=LEARNING_RATE, watch=True,
    )
    results_df["val_fold_id"] = fold_id
    fold_results.append(results_df)
    for eval_group, m in metrics_by_group.items():
        fold_metrics_rows.append({"val_fold_id": fold_id, "eval_group": eval_group, **m})

    fit_m, val_m = metrics_by_group["fit"], metrics_by_group["val"]
    print(f"fold {fold_id}: "
          f"fit(accuracy={fit_m['accuracy']:.3f}, f1={fit_m['f1']:.3f}, auroc={fit_m['auroc']:.3f}) | "
          f"val(accuracy={val_m['accuracy']:.3f}, f1={val_m['f1']:.3f}, auroc={val_m['auroc']:.3f}) "
          f"(fit={len(row_groups['fit'])}, val={len(row_groups['val'])})")

    wandb.log({
        "fold": fold_id,
        "fit/accuracy": fit_m["accuracy"], "fit/f1": fit_m["f1"], "fit/auroc": fit_m["auroc"],
        "val/accuracy": val_m["accuracy"], "val/f1": val_m["f1"], "val/auroc": val_m["auroc"],
        "n_fit": len(row_groups["fit"]), "n_val": len(row_groups["val"]),
    }, step=fold_id)

cv_results_df = pd.concat(fold_results, ignore_index=True)
fold_metrics_df = pd.DataFrame(fold_metrics_rows)

print("\nmean +/- std across folds:")
summary_df = fold_metrics_df.groupby("eval_group")[["accuracy", "f1", "auroc"]].agg(["mean", "std"])
print(summary_df)

for eval_group in summary_df.index:
    for metric in ["accuracy", "f1", "auroc"]:
        run.summary[f"{eval_group}_{metric}_mean"] = summary_df.loc[eval_group, (metric, "mean")]
        run.summary[f"{eval_group}_{metric}_std"] = summary_df.loc[eval_group, (metric, "std")]

results_artifact = wandb.Artifact(name="cv_results", type="results")
results_artifact.add(wandb.Table(dataframe=cv_results_df), "cv_results")
logged_results_artifact = run.log_artifact(results_artifact)
# logged_results_artifact.wait()  # block until server-side commit, so the round-trip
                                 # fetch right below doesn't race the async upload

cv_results_df.head()

wandb: logging graph, to disable use `wandb.watch(log_graph=False)`


fold 0: fit(accuracy=0.839, f1=0.901, auroc=0.895) | val(accuracy=0.780, f1=0.859, auroc=0.781) (fit=13804, val=3239)


wandb: logging graph, to disable use `wandb.watch(log_graph=False)`


fold 1: fit(accuracy=0.778, f1=0.855, auroc=0.885) | val(accuracy=0.719, f1=0.811, auroc=0.813) (fit=13700, val=3343)


wandb: logging graph, to disable use `wandb.watch(log_graph=False)`


...

wandb: logging graph, to disable use `wandb.watch(log_graph=False)`


fold 3: fit(accuracy=0.824, f1=0.890, auroc=0.892) | val(accuracy=0.698, f1=0.793, auroc=0.812) (fit=13468, val=3575)


wandb: logging graph, to disable use `wandb.watch(log_graph=False)`


fold 4: fit(accuracy=0.809, f1=0.879, auroc=0.871) | val(accuracy=0.776, f1=0.859, auroc=0.822) (fit=13416, val=3627)

mean +/- std across folds:
            accuracy                  f1               auroc          
                mean       std      mean       std      mean       std
eval_group                                                            
fit         0.813577  0.022478  0.881752  0.016750  0.887215  0.010028
val         0.759810  0.051214  0.844013  0.041908  0.790875  0.039044


,sequence,fold_id,peptide_id,smiles,organism,has_noncanonical,label,true_label_int,pred_label_int,logit,prob,correct,eval_group,val_fold_id
0,LFIFFF,4,10,CC[C@H](C)[C@H](NC(=O)[C@H](Cc1ccccc1)NC(=O)[C...,Staphylococcus aureus,False,active,1,0,-0.584151,0.357978,False,fit,0
1,ENREVPPGFTALIKTLRKCKII,3,16,CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCCN...,Escherichia coli,False,active,1,1,1.694927,0.844871,True,fit,0
2,ENREVPPGFTALIKTLRKCKII,3,16,CC[C@H](C)[C@H](NC(=O)[C@@H](NC(=O)[C@H](CCCCN...,Staphylococcus aureus,False,active,1,1,0.047834,0.511956,True,fit,0
3,GMASKAGAIAGKIAKVALKAL,3,37,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C@...,Escherichia coli,False,active,1,1,0.997025,0.730473,True,fit,0
4,GMASKAGAIAGKIAKVALKAL,3,37,CC[C@H](C)[C@H](NC(=O)[C@H](C)NC(=O)CNC(=O)[C@...,Staphylococcus aureus,False,active,1,0,-0.879842,0.293211,False,fit,0


In [17]:
from sklearn.metrics import classification_report

analytics_df = cv_results_df.loc[cv_results_df["has_noncanonical"] == True]

In [18]:
for fold in analytics_df["val_fold_id"].unique():
    train_query = analytics_df[(analytics_df["val_fold_id"] == fold) & (analytics_df["eval_group"] == "fit")]
    val_query = analytics_df[(analytics_df["val_fold_id"] == fold) & (analytics_df["eval_group"] == "val")]

    print(f"fold {fold}:")
    print(f"***train_query: {len(train_query)}***")
    print(classification_report(train_query["true_label_int"], train_query["pred_label_int"]))
    print(f"***val_query: {len(val_query)}***")
    print(classification_report(val_query["true_label_int"], val_query["pred_label_int"]))

fold 0:
***train_query: 2961***
              precision    recall  f1-score   support

           0       0.57      0.82      0.68       322
           1       0.98      0.93      0.95      2639

    accuracy                           0.91      2961
   macro avg       0.78      0.87      0.81      2961
weighted avg       0.93      0.91      0.92      2961

***val_query: 444***
              precision    recall  f1-score   support

           0       0.52      0.90      0.66       105
           1       0.96      0.75      0.84       339

    accuracy                           0.78       444
   macro avg       0.74      0.82      0.75       444
weighted avg       0.86      0.78      0.80       444

fold 1:
***train_query: 2905***
              precision    recall  f1-score   support

           0       0.45      0.87      0.60       367
           1       0.98      0.85      0.91      2538

    accuracy                           0.85      2905
   macro avg       0.72      0.86      0.75

In [ ]:
import sys

# Add the parent directory of the 'soamp' package to sys.path.
# If the package 'soamp' is located at '/content/soamp/src/soamp',
# then '/content/soamp/src' needs to be on sys.path for 'import soamp' to work.
if '/content/soamp/src' not in sys.path:
    sys.path.insert(0, '/content/soamp/src')

# Clear any cached 'soamp' modules from sys.modules to force a fresh import.
# This helps resolve issues where previous failed imports are cached.
for module_name in list(sys.modules.keys()):
    if module_name == 'soamp' or module_name.startswith('soamp.'):
        del sys.modules[module_name]